# Level 7: Quantum Fourier Transform (QFT)
## Qiskit Edition

In this notebook, you will:

1. Understand the **Discrete Fourier Transform** and its quantum counterpart
2. Build **QFT from scratch** using controlled phase gates
3. Implement the **inverse QFT**
4. See QFT as the foundation for Shor's algorithm and phase estimation

In [ ]:
from qiskit import QuantumCircuit
from qiskit_aer import AerSimulator
from qiskit.visualization import plot_histogram, plot_bloch_multivector
from qiskit.quantum_info import Statevector
import matplotlib.pyplot as plt
import numpy as np

simulator = AerSimulator()
print("Ready!")

---
## 1. From DFT to QFT

### Classical DFT
The Discrete Fourier Transform converts a vector of $N$ amplitudes from the computational basis to the frequency basis:

$$y_k = \frac{1}{\sqrt{N}} \sum_{j=0}^{N-1} x_j \, \omega^{jk}$$

where $\omega = e^{2\pi i / N}$ is the $N$-th root of unity.

### Quantum QFT
The QFT does the same transformation on quantum amplitudes:

$$\text{QFT}|j\rangle = \frac{1}{\sqrt{N}} \sum_{k=0}^{N-1} \omega^{jk} |k\rangle$$

**Key advantage**: Classical FFT takes $O(N \log N)$ operations. QFT takes $O(n^2)$ gates, where $N = 2^n$. This is an exponential speedup!

### QFT Circuit Structure
The QFT circuit uses:
1. **Hadamard gates** (H)
2. **Controlled phase rotation gates** ($CR_k$) where $R_k = \begin{pmatrix} 1 & 0 \\ 0 & e^{2\pi i/2^k} \end{pmatrix}$
3. **Swap gates** at the end to reverse bit order

In [ ]:
def qft_circuit(n_qubits):
    """Build QFT circuit from scratch."""
    qc = QuantumCircuit(n_qubits)
    
    for i in range(n_qubits):
        # Hadamard on qubit i
        qc.h(i)
        
        # Controlled rotations
        for j in range(i + 1, n_qubits):
            # CR_k where k = j - i + 1
            k = j - i + 1
            angle = 2 * np.pi / (2**k)
            qc.cp(angle, j, i)  # Controlled-Phase
    
    # Swap qubits to reverse bit order
    for i in range(n_qubits // 2):
        qc.swap(i, n_qubits - 1 - i)
    
    return qc

# Build 3-qubit QFT
qft3 = qft_circuit(3)
print("3-Qubit QFT Circuit:")
qft3.draw('mpl')

In [ ]:
# 4-qubit QFT
qft4 = qft_circuit(4)
print("4-Qubit QFT Circuit:")
qft4.draw('mpl')

---
## 2. QFT in Action

Let's apply QFT to different input states and see what happens.

In [ ]:
# QFT of |000> — should give uniform superposition
qc = QuantumCircuit(3)
qc.append(qft_circuit(3), range(3))

sv = Statevector.from_instruction(qc)
print("QFT|000>:")
print("Amplitudes:", np.round(sv.data, 3))
print("\nAll equal — uniform superposition!")

In [ ]:
# QFT of |001>
qc = QuantumCircuit(3)
qc.x(0)  # |001>
qc.append(qft_circuit(3), range(3))

sv = Statevector.from_instruction(qc)
print("QFT|001>:")
print("Amplitudes:")
for i, amp in enumerate(sv.data):
    print(f"  |{i:03b}>: {amp:.4f}  (|amp|² = {abs(amp)**2:.4f})")

In [ ]:
# Compare QFT on all 8 basis states
fig, axes = plt.subplots(2, 4, figsize=(16, 8))
axes = axes.flatten()

for state_idx in range(8):
    qc = QuantumCircuit(3)
    # Prepare basis state
    for bit in range(3):
        if (state_idx >> bit) & 1:
            qc.x(bit)
    qc.append(qft_circuit(3), range(3))
    
    sv = Statevector.from_instruction(qc)
    probs = np.abs(sv.data)**2
    
    axes[state_idx].bar(range(8), probs, color='teal')
    axes[state_idx].set_title(f'QFT|{state_idx:03b}>')
    axes[state_idx].set_ylim(0, 0.5)
    axes[state_idx].set_xlabel('State')

plt.tight_layout()
plt.show()
print("Notice: QFT gives uniform probabilities — the information is in the PHASES!")

---
## 3. Inverse QFT

The inverse QFT converts back from the frequency basis. It's essential for algorithms like phase estimation.

In [ ]:
def inverse_qft_circuit(n_qubits):
    """Build inverse QFT = QFT†"""
    return qft_circuit(n_qubits).inverse()

# Verify: QFT followed by QFT† should give back the original state
qc = QuantumCircuit(3, 3)
qc.x(0)  # Start with |001>
qc.x(2)  # Now |101>
qc.barrier()
qc.append(qft_circuit(3), range(3))     # Apply QFT
qc.barrier()
qc.append(inverse_qft_circuit(3), range(3))  # Apply QFT†
qc.measure(range(3), range(3))

result = simulator.run(qc, shots=1000).result().get_counts()
print(f"Original: |101>")
print(f"After QFT → QFT†: {result}")
print("State perfectly recovered!")

---
## 4. QFT for Period Finding (Preview)

The real power of QFT is in finding **periodicity**. If a function has period $r$, applying QFT to the function values reveals peaks at multiples of $N/r$.

This is the key to **Shor's algorithm** for factoring!

In [ ]:
# Demo: QFT on a periodic state
# Create a state with period 2: |000> + |010> + |100> + |110>
qc = QuantumCircuit(3)
qc.h(0)  # Superposition of bit 0
qc.h(1)  # Superposition of bit 1
# Qubit 2 stays |0> → period 2 in the state pattern

qc.append(qft_circuit(3), range(3))
qc.measure_all()

result = simulator.run(qc, shots=4000).result().get_counts()
print("QFT of periodic state (period 2):")
print(result)
plot_histogram(result, title="QFT reveals periodicity")

---
## 5. Key Takeaways

| Concept | Description |
|---------|-------------|
| **QFT** | Quantum Fourier Transform: $O(n^2)$ vs classical $O(N \log N)$ |
| **Controlled-$R_k$** | The key building block: phase rotations |
| **Phase encoding** | QFT encodes information in phases, not probabilities |
| **Period finding** | QFT reveals periodicity — foundation of Shor's algorithm |

---
## 6. Challenges

1. **5-qubit QFT**: Build the 5-qubit QFT circuit. How many gates does it use?
2. **Phase visualization**: Use `plot_bloch_multivector` to see the phases after QFT on |011>.
3. **QFT addition**: Implement quantum addition in the Fourier basis (add two numbers encoded in qubits).

In [ ]:
# Your challenge code here!

---
**Next up: [Level 8 — Phase Estimation & Shor's Algorithm](../Level_08_Phase_Estimation_Shor/Level_08_Qiskit.ipynb)**